======================================================================================================

$$\text{Matched feedback/constant-flow robustness grid for hydraulic impairment. This is a 
diagnostic sweep, not calibration to experimental observations}$$
$$\text{Created by Avulundiah Edwin Phiri}$$
$$\text{Department of Mathematics, Copperbelt University, Kitwe, Zambia}$$
$$ Date: 30/07/2026$$

======================================================================================================

In [6]:
"""Matched feedback/constant-flow robustness grid for hydraulic impairment.

This is a diagnostic sweep, not calibration to experimental observations.
Run: python Figures/feedback_robustness_grid.py
Requires NumPy, SciPy and Matplotlib. All outputs are written next to the script.

For each scenario the two ODE systems differ only in substrate replenishment.
Both have identical positive mineral seeds, initial substrate and diagnostics.
Pore closure is a stopping event. The comparison excludes pairs in which
either model encounters the closure boundary before the end of follow-up.
"""

from csv import DictReader, DictWriter
from pathlib import Path
import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, minimize_scalar


ROOT = Path(__file__).resolve().parent
DELTA_GRID = np.linspace(0.08, 0.50, 15)
LAMBDA_GRID = np.linspace(0.15, 0.85, 21)
N_GRID = (2.0, 5.0, 9.0)
X0_GRID = (0.05, 0.20, 0.40)
KAPPA, GAMMA = 0.30, 1.0
S0, EPSILON, T_END = 0.90, 0.40, 30.0
RTOL, ATOL = 2e-9, 2e-11


def rhs(time, state, delta, removal, exponent, initial_loading, feedback):
    s, x = state
    growth = s / (KAPPA + s)
    exchange = (1.0 + initial_loading - x) ** exponent if feedback else 1.0
    return (
        delta * exchange * (1.0 - s) - GAMMA * growth * x,
        (growth - removal) * x,
    )


def pore_closure(time, state, delta, removal, exponent, initial_loading, feedback):
    return 1.0 + initial_loading - state[1]


pore_closure.terminal = True
pore_closure.direction = -1.0


def simulate(delta, removal, exponent, initial_loading, feedback):
    args = (delta, removal, exponent, initial_loading, feedback)
    sol = solve_ivp(
        rhs, (0.0, T_END), (S0, initial_loading), args=args,
        events=pore_closure, method="DOP853",
        dense_output=True, rtol=RTOL, atol=ATOL,
    )
    if not sol.success:
        return {"status": "solver_failure"}
    if sol.t_events[0].size:
        return {"status": "pore_closure", "closure_time": float(sol.t_events[0][0])}

    def permeability(t):
        return (1.0 + initial_loading - sol.sol(t)[1]) ** exponent

    tgrid = np.linspace(0.0, T_END, 1201)
    kgrid = permeability(tgrid)
    kminus = kgrid - EPSILON
    crossing = [
        float(brentq(lambda t: permeability(t) - EPSILON, a, b, xtol=1e-10))
        for a, b, fa, fb in zip(tgrid[:-1], tgrid[1:], kminus[:-1], kminus[1:])
        if fa * fb < 0
    ]
    boundaries = [0.0] + crossing + [T_END]
    duration = sum(
        b - a for a, b in zip(boundaries[:-1], boundaries[1:])
        if permeability((a + b) / 2.0) <= EPSILON
    )
    j = int(np.argmin(kgrid))
    left = float(tgrid[max(j - 1, 0)])
    right = float(tgrid[min(j + 1, len(tgrid) - 1)])
    if left < right:
        minimum = minimize_scalar(permeability, bounds=(left, right), method="bounded")
        kmin = min(float(minimum.fun), float(kgrid[j]))
    else:
        kmin = float(kgrid[j])
    down = crossing[0] if crossing else float("nan")
    recovery = crossing[1] if len(crossing) > 1 else float("nan")
    return {
        "status": "full",
        "kmin": kmin,
        "duration": duration,
        "first_down": down,
        "first_recovery": recovery,
        "below_at_horizon": int(permeability(T_END) <= EPSILON),
    }


def summarize(rows):
    print(f"Total paired scenarios: {len(rows)}")
    valid = [r for r in rows if r["pair_status"] == "full"]
    print(f"Full-horizon admissible comparisons: {len(valid)}")
    for label, key in (("Feedback closure", "feedback_status"),
                       ("Constant-flow closure", "constant_status")):
        print(f"{label}: {sum(r[key] == 'pore_closure' for r in rows)}")
    for x0 in X0_GRID:
        subset = [r for r in valid if r["x0"] == x0]
        p_k = sum(r["delta_kmin"] > 1e-6 for r in subset)
        m_k = sum(r["delta_kmin"] < -1e-6 for r in subset)
        z_k = len(subset) - p_k - m_k
        p_t = sum(r["delta_duration"] > 1e-6 for r in subset)
        m_t = sum(r["delta_duration"] < -1e-6 for r in subset)
        z_t = len(subset) - p_t - m_t
        print(
            f"x0={x0:.2f}: N={len(subset)}, Delta kmin + / - / 0 = "
            f"{p_k}/{m_k}/{z_k}; Delta T_eps + / - / 0 = {p_t}/{m_t}/{z_t}; "
            f"mean Delta T_eps={np.mean([r['delta_duration'] for r in subset]):.4f}"
        )
    for n in N_GRID:
        subset = [r for r in valid if r["n"] == n and r["x0"] == 0.20]
        print(
            f"x0=0.20 n={n:g}: valid={len(subset)}, "
            f"min/max Delta T_eps="
            f"{min(r['delta_duration'] for r in subset):.4f}/"
            f"{max(r['delta_duration'] for r in subset):.4f}"
        )
    return valid


def heatmap(rows):
    fig, axes = plt.subplots(1, len(N_GRID), figsize=(9.6, 3.25), sharey=True)
    matrices = []
    for n in N_GRID:
        mat = np.full((len(LAMBDA_GRID), len(DELTA_GRID)), np.nan)
        for r in rows:
            if r["n"] == n and r["x0"] == 0.20 and r["pair_status"] == "full":
                ii = int(np.argmin(abs(LAMBDA_GRID - r["lambda"])))
                jj = int(np.argmin(abs(DELTA_GRID - r["delta"])))
                mat[ii, jj] = r["delta_duration"]
        matrices.append(mat)
    magnitude = max(np.nanmax(abs(matrix)) for matrix in matrices)
    magnitude = max(magnitude, 0.05)
    cmap = plt.get_cmap("RdBu").copy()
    cmap.set_bad("#bcbcbc")
    norm = TwoSlopeNorm(vcenter=0.0, vmin=-magnitude, vmax=magnitude)
    for ax, n, mat in zip(axes, N_GRID, matrices):
        mesh = ax.pcolormesh(
            DELTA_GRID, LAMBDA_GRID, np.ma.masked_invalid(mat),
            cmap=cmap, norm=norm, shading="nearest", rasterized=True,
        )
        ax.axhline(1 / (1 + KAPPA), color="black", ls=":", lw=1)
        ax.set_xlabel(r"Replenishment $\delta$")
        ax.set_title(rf"$n={n:g}$", fontsize=11)
        ax.tick_params(labelsize=9)
    axes[0].set_ylabel(r"Removal $\lambda$")
    fig.subplots_adjust(left=.075, right=.86, bottom=.19, top=.90, wspace=.10)
    color_axis = fig.add_axes([.887, .20, .023, .67])
    cb = fig.colorbar(mesh, cax=color_axis)
    cb.set_label(r"$\Delta T_{\varepsilon}=T_{\varepsilon}^{CF}-T_{\varepsilon}^{FB}$")
    fig.savefig(ROOT / "feedback_robustness_heatmap.pdf", bbox_inches="tight")
    fig.savefig(ROOT / "feedback_robustness_heatmap.png", dpi=450, bbox_inches="tight")
    plt.close(fig)


def main():
    if "--plot-only" in sys.argv:
        with (ROOT / "feedback_robustness_results.csv").open(newline="") as stream:
            rows = list(DictReader(stream))
        for row in rows:
            for key in ("delta", "lambda", "n", "x0", "delta_duration"):
                if row.get(key):
                    row[key] = float(row[key])
        heatmap(rows)
        print("Replotted feedback_robustness_heatmap.pdf/png from saved CSV")
        return
    rows = []
    for x0 in X0_GRID:
        for n in N_GRID:
            for lam in LAMBDA_GRID:
                for delta in DELTA_GRID:
                    fb = simulate(delta, lam, n, x0, True)
                    cf = simulate(delta, lam, n, x0, False)
                    pair_status = "full" if (
                        fb["status"] == "full" and cf["status"] == "full"
                    ) else "censored"
                    row = {
                        "delta": float(delta), "lambda": float(lam),
                        "n": n, "x0": x0,
                        "R0": 1.0 / (lam * (1.0 + KAPPA)),
                        "feedback_status": fb["status"],
                        "constant_status": cf["status"],
                        "pair_status": pair_status,
                    }
                    if pair_status == "full":
                        row.update({
                            "kmin_FB": fb["kmin"], "kmin_CF": cf["kmin"],
                            "T_epsilon_FB": fb["duration"],
                            "T_epsilon_CF": cf["duration"],
                            "delta_kmin": fb["kmin"] - cf["kmin"],
                            "delta_duration": cf["duration"] - fb["duration"],
                            "FB_first_recovery": fb["first_recovery"],
                            "CF_first_recovery": cf["first_recovery"],
                            "FB_below_T": fb["below_at_horizon"],
                            "CF_below_T": cf["below_at_horizon"],
                        })
                    rows.append(row)
    valid = summarize(rows)
    if not valid:
        raise RuntimeError("No complete admissible comparisons")
    fieldnames = sorted({key for row in rows for key in row})
    with (ROOT / "feedback_robustness_results.csv").open("w", newline="") as stream:
        writer = DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    heatmap(rows)
    print("Created: feedback_robustness_results.csv, feedback_robustness_heatmap.pdf/png")


if __name__ == "__main__":
    main()


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/user-data/outputs/clogging_design_chart.pdf'